In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Creating subagents

In [4]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [6]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    model='gpt-5-nano',
    tools=[square_root]
)

subagent_2 = create_agent(
    model='gpt-5-nano',
    tools=[square]
)

## Calling subagents

In [8]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model='gpt-5-nano',
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [13]:
question = "ریشه دوم 456 چه میشود؟"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})
print(response["messages"][-1].content)

ریشه دوم عدد 456 برابر است با تقریب زیر:
- حدوداً 21.354156504062622
- با گردکردن به 6 رقم اعشار: 21.354157


In [14]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='ریشه دوم 456 چه میشود؟', additional_kwargs={}, response_metadata={}, id='7cbdb555-e1bb-40a3-af30-e0450cfc9d61'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 346, 'prompt_tokens': 201, 'total_tokens': 547, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': None, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DqaIs5A9OrMnq0RFOaCvu1TiilTSU', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec533-fca8-78b0-a635-a365bc3d9b09-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': 'call_WpKitBNWwSdnh53ZJNRs5w1O', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 201, 'output_tokens': 346, 'total_